Gerne, hier sind die Aufgabenstellungen aus dem `temperature.pdf` in vereinfachtem Deutsch zusammengefasst:

**Aufgabe 1: Modell `temperature_nd.mzn`**

*   **Ziel:** Bestimme die Gebäudetemperatur über die Zeit, ohne Eingriffe.
*   **Eingabe:**
    *   Eine Starttemperatur für das Gebäude (`start`), die zwischen 25 und 30 Grad liegt.
    *   Eine Abfolge von stündlichen Außentemperaturen (`readings`).
*   **Berechnung der Temperatur ohne Eingriff:** Die Gebäudetemperatur bewegt sich auf den Durchschnitt der aktuellen Gebäudetemperatur und der Außentemperatur zu, abgerundet.
*   **Ausgabe:** Ein Array der Gebäudetemperaturen, beginnend mit der Starttemperatur, für jede Stunde.
*   **Beispiel (Eingabe):** `start=25; readings=;`
*   **Beispiel (Erwartete Ausgabe):** `temp = ;`

**Aufgabe 2: Modell `temperature.mzn`**

*   **Ziel:** Ermittle die **minimalen Kosten**, um die Gebäudetemperatur **durchgehend zwischen 25 und 30 Grad** zu halten.
*   **Verfügbare Aktionen (mit Kosten):**
    *   **Heizen (`heat`):** Erhöht die Temperatur um 1 Grad. Kosten: 1$.
    *   **Stark heizen (`strongly_heat`):** Erhöht die Temperatur um 4 Grad. Kosten: 5$.
    *   **Kühlen (`cool`):** Senkt die Temperatur um 2 Grad. Kosten: 3$.
    *   **Stark kühlen (`strongly_cool`):** Senkt die Temperatur um 5 Grad. Kosten: 9$.
    *   **Nichts tun (`do_nothing`):** Keine Kosten.
*   **Ausgabe:**
    *   Die tatsächlichen Temperaturen zu jeder Stunde.
    *   Die getroffenen Entscheidungen für jede Stunde.
    *   Die **Gesamtkosten**.
*   **Beispiel (Eingabe wie in Aufgabe 1):** `start=25; readings=;`
*   **Beispiel (Optimale Lösung):**
    *   `temp = ;`
    *   `choice = [do_nothing, cool, do_nothing, strongly_heat, strongly_heat];`
    *   `cost = 13;`
*   **Zusätzliche Tests:** Das Modell soll mit weiteren Datensätzen (als `.dzn`-Dateien bereitgestellt) getestet werden.

**Technische Anforderung:**

*   Für die Bearbeitung ist MiniZinc 2.2.x erforderlich.

In [ ]:
''' A1
Die Starttemperatur ist 25. Die Gebäude Soll-Temperatur wird nacheinander eingestellt auf
folgende Werte [35, 35, 20, 20, 20]. Die neue Gebäude Ist-Temperatur ist der abgerundete Mittelwert
zwischen der neuen Soll-Temperatur und der alten Ist-Temperatur. Berechne die Abfolge der Ist-Temperaturen.
'''

int: start = 25;
array[1..5] of int: soll = [35, 35, 20, 20, 20];

array[0..5] of var int: ist;           % decision Variables

constraint ist[0] = start;

constraint
  forall(i in 1..5)(
    ist[i] = floor((ist[i-1] + soll[i]) / 2)
  );

solve satisfy;

output ["Ist-Temperaturen: \(ist)"];


In [ ]:
''' A2
Die Starttemperatur ist 25. Zur Regelung der Ist-Temperaturen stehen folgende Aktionen 
zur Verfügung, die die Temperatur um den angegebenen Wert verändern:

Aktion     Änderung  Kosten
heat           +1     1
strongly_heat  +4     5
cool           -2     3
strongly_cool  -5     9
do_nothing      0     0

Die Gebäude Soll-Temperatur wird nacheinander eingestellt auf die 
Werte [35, 35, 20, 20, 20].
Neue Ist-Temperatur = (alte Ist-Temperatur + Solltemperatur)//2 + Aktion

Durch welche Aktionen können bei minimalen Kosten die Ist-Temperaturen
in einem Bereich zwischen 25 und 30 gehalten werden.

'''

In [ ]:
int: start = 25;
array[1..5] of int: soll = [35, 35, 20, 20, 20];
int: N = length(soll);
enum ACTIONS = {heat, strongly_heat, cool, strongly_cool, do_nothing};
array[ACTIONS] of int: aenderung = [1, 4, -2, -5, 0];
array[ACTIONS] of int: kosten = [1, 5, 3, 9, 0];


array[0..N] of var 25..30: tmp;                     % decision variables
array[1..N] of var ACTIONS: wahl;
var int: cost = sum(t in 1..N)(kosten[wahl[t]]);

constraint tmp[0] = start;
constraint forall (t in 1..N) (tmp[t] = (tmp[t-1]+soll[t]) div 2 + aenderung[wahl[t]]);
output ["temp = \(tmp) \nwahl = \(wahl)\nkosten = \(cost)"];
solve minimize cost;